<a href="https://colab.research.google.com/github/zarduya/SQL-DB-PROJECT/blob/main/sql_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ayrus Pharmacy: Randomly Generated Retail Database
This notebook generates a realistic synthetic SQLite database for a fictional UK
health and beauty retailer, for my 7PAM2005 Data Mining & Discovery SQL assignment.
All data is randomly generated; no real customer data is used.

In [1]:
!pip install faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 7.8 MB/s  0:00:00


In [2]:
import sqlite3
import random
import numpy as np
import pandas as pd
from faker import Faker

In [3]:
# set the seed so the random data comes out identical every time we run the notebook
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_GB')   # for british names
Faker.seed(SEED)

In [4]:
from datetime import date

# some fixed variables ?
BRAND = "Ayrus Pharmacy"
OWN_BRAND = "Ayrus"
N_PRODUCTS = 250
DATA_START = date(2025, 1, 1)
DATA_END = date(2025, 12, 31)

# regions get their own table so that each name is typed only once
regions_df = pd.DataFrame({
    'region_id': [1, 2, 3, 4, 5],
    'region_name': ['London', 'South East', 'East of England',
                    'West Midlands', 'North West']
})
regions_df

,region_id,region_name
0,1,London
1,2,South East
2,3,East of England
3,4,West Midlands
4,5,North West


In [5]:
# stores are typed in manually since a real company already knows where its stores are
# format is ordinal: Local < High Street < Flagship
STORE_LIST = [
    ('Oxford Street', 'London', 1, 'Flagship'),
    ('Croydon', 'London', 1, 'High Street'),
    ('Stratford', 'London', 1, 'High Street'),
    ('Brighton', 'Brighton', 2, 'High Street'),
    ('Reading', 'Reading', 2, 'High Street'),
    ('Oxford', 'Oxford', 2, 'Local'),
    ('Luton', 'Luton', 3, 'High Street'),
    ('Cambridge', 'Cambridge', 3, 'High Street'),
    ('Watford', 'Watford', 3, 'Local'),
    ('Norwich', 'Norwich', 3, 'Local'),
    ('Birmingham Bullring', 'Birmingham', 4, 'Flagship'),
    ('Coventry', 'Coventry', 4, 'Local'),
    ('Manchester Arndale', 'Manchester', 5, 'Flagship'),
    ('Liverpool', 'Liverpool', 5, 'High Street'),
    ('Preston', 'Preston', 5, 'Local'),
]

# typical floor size for each format as (average, spread) in m²
# normal distribution so stores of the same format are similar but not identical
SIZE_BY_FORMAT = {'Local': (150, 30), 'High Street': (400, 80), 'Flagship': (1200, 200)}

stores = []
for store_id, (label, city, region_id, fmt) in enumerate(STORE_LIST, start=1):
    mean, sd = SIZE_BY_FORMAT[fmt]
    size = round(max(np.random.normal(mean, sd), 50), 1)   # never smaller than 50 m²
    opened = fake.date_between(start_date=date(1985, 1, 1), end_date=date(2022, 12, 31))
    stores.append({
        'store_id': store_id,
        'store_name': f"{BRAND} {label}",
        'city': city,
        'region_id': region_id,            # foreign key to regions
        'store_format': fmt,               # ordinal
        'size_sq_m': size,                 # ratio, 0 m² would mean no space at all
        'opened_date': opened.isoformat()  # interval, saved as 'YYYY-MM-DD' text since SQLite has no date type
    })

stores_df = pd.DataFrame(stores)
stores_df

,store_id,store_name,city,region_id,store_format,size_sq_m,opened_date
0,1,Ayrus Pharmacy Oxford Street,London,1,Flagship,1299.3,2009-04-18
1,2,Ayrus Pharmacy Croydon,London,1,High Street,388.9,1985-12-14
2,3,Ayrus Pharmacy Stratford,London,1,High Street,451.8,1995-06-14
3,4,Ayrus Pharmacy Brighton,Brighton,2,High Street,521.8,1993-06-25
4,5,Ayrus Pharmacy Reading,Reading,2,High Street,381.3,2012-12-25
5,6,Ayrus Pharmacy Oxford,Oxford,2,Local,143.0,2010-09-18
6,7,Ayrus Pharmacy Luton,Luton,3,High Street,526.3,2018-11-25
7,8,Ayrus Pharmacy Cambridge,Cambridge,3,High Street,461.4,1988-04-21
8,9,Ayrus Pharmacy Watford,Watford,3,Local,135.9,2001-01-12
9,10,Ayrus Pharmacy Norwich,Norwich,3,Local,166.3,1986-02-18
